# 🕸️ LangGraph — Complete Guide
### Building Stateful Graph-Based AI Agents

---

**Topics Covered:**
1. Installation & Setup
2. Core Concepts — `StateGraph` & `TypedDict`
3. Nodes — The Building Blocks
4. Edges & Conditional Routing
5. Compiling & Running the Graph
6. Checkpointing & Persistence
7. Human-in-the-Loop
8. Streaming Agent Outputs
9. Multi-Agent Supervisor Pattern
10. Prebuilt ReAct Agent (`create_react_agent`)
11. Debugging & Observability

---
## 1. Installation & Setup

In [ ]:
# Install required packages
!pip install langgraph langchain-anthropic langchain-core -q

In [ ]:
import os
from getpass import getpass

# Set your Anthropic API key
if not os.environ.get("ANTHROPIC_API_KEY"):
    os.environ["ANTHROPIC_API_KEY"] = getpass("Enter your Anthropic API key: ")

print("✅ API key configured")

---
## 2. Core Concepts — `StateGraph` & `TypedDict`

The foundation of every LangGraph application is a **StateGraph** — a typed shared state that flows through every node in the graph.

**Key concepts:**
- `TypedDict` — defines the schema of the state
- `Annotated` + `operator.add` — reducer that appends to lists instead of overwriting
- `StateGraph(State)` — the graph container

In [ ]:
from typing import TypedDict, Annotated, Sequence
from langchain_core.messages import BaseMessage
import operator

# Define the shared state schema
class AgentState(TypedDict):
    # Annotated with operator.add → messages are APPENDED, not overwritten
    messages: Annotated[Sequence[BaseMessage], operator.add]
    # Plain fields are overwritten on each update
    next_step: str
    iteration_count: int

print("✅ AgentState schema defined")
print("Fields:", list(AgentState.__annotations__.keys()))

In [ ]:
from langgraph.graph import StateGraph, START, END

# Initialise the graph with our state schema
graph_builder = StateGraph(AgentState)

print("✅ StateGraph initialised")
print("Graph type:", type(graph_builder))

---
## 3. Nodes — The Building Blocks

A **node** is a Python function that:
- Receives the current state as input
- Performs work (call LLM, call tool, process data)
- Returns a **partial** state update (only the keys it changed)

In [ ]:
from langchain_anthropic import ChatAnthropic
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, AIMessage
from langgraph.prebuilt import ToolNode

# ── Define Tools ──────────────────────────────────────
@tool
def get_weather(location: str) -> str:
    """Get current weather for a location."""
    data = {
        "Singapore": "32°C, Humid",
        "London": "12°C, Rainy",
        "Tokyo": "22°C, Clear"
    }
    return data.get(location, f"No data for {location}")

@tool
def calculate(expression: str) -> str:
    """Safely evaluate a mathematical expression."""
    try:
        allowed = set('0123456789+-*/()., ')
        if all(c in allowed for c in expression):
            return str(eval(expression))
        return "Invalid expression"
    except Exception as e:
        return f"Error: {e}"

tools = [get_weather, calculate]

# ── Initialise Model ──────────────────────────────────
model = ChatAnthropic(model="claude-sonnet-4-5")
model_with_tools = model.bind_tools(tools)

# ── Agent Node ────────────────────────────────────────
def agent_node(state: AgentState) -> dict:
    """LLM decides what to do next based on message history."""
    response = model_with_tools.invoke(state["messages"])
    count = state.get("iteration_count", 0) + 1
    return {
        "messages": [response],
        "iteration_count": count
    }

# ── Tool Node (prebuilt) ───────────────────────────────
# Automatically executes all tool_calls from the last AIMessage
tool_node = ToolNode(tools=tools)

print("✅ Nodes defined: agent_node, tool_node")

---
## 4. Edges & Conditional Routing

**Edges** determine how control flows between nodes:
- `add_edge(A, B)` — always go from A to B
- `add_conditional_edges(A, router_fn)` — call a function to decide where to go

In [ ]:
from langgraph.graph import StateGraph, START, END

# ── Router Function ───────────────────────────────────
def should_continue(state: AgentState) -> str:
    """Inspect the last message — if tool calls exist, go to tools; else END."""
    last_message = state["messages"][-1]
    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return "tools"
    return END

# ── Build the Graph ───────────────────────────────────
graph_builder = StateGraph(AgentState)

# Add nodes
graph_builder.add_node("agent", agent_node)
graph_builder.add_node("tools", tool_node)

# Add edges
graph_builder.add_edge(START, "agent")          # always start at agent
graph_builder.add_edge("tools", "agent")         # after tools → loop back to agent

# Add conditional edge: agent → tools OR END
graph_builder.add_conditional_edges(
    "agent",
    should_continue,
    {"tools": "tools", END: END}
)

print("✅ Graph structure defined")
print("Nodes:", list(graph_builder.nodes.keys()))

---
## 5. Compiling & Running the Graph

`graph.compile()` validates the graph and returns a `CompiledGraph` — the runnable application.

In [ ]:
# Compile without checkpointing (no memory between runs)
app = graph_builder.compile()

print("✅ Graph compiled successfully")

# Visualise the graph structure (ASCII)
print("\nGraph structure:")
print(app.get_graph().draw_ascii())

In [ ]:
# Run the graph
initial_state = {
    "messages": [HumanMessage(content="What is the weather in Singapore and calculate 99 * 3?")],
    "next_step": "",
    "iteration_count": 0
}

result = app.invoke(initial_state)

print("Final answer:", result["messages"][-1].content)
print("Iterations:", result["iteration_count"])

In [ ]:
# Trace all messages in the run
print("=" * 60)
print("FULL EXECUTION TRACE")
print("=" * 60)
for i, msg in enumerate(result["messages"]):
    role = type(msg).__name__
    content = msg.content if isinstance(msg.content, str) else str(msg.content)[:200]
    print(f"\n[{i+1}] {role}: {content[:300]}")

---
## 6. Checkpointing & Persistence

A **checkpointer** persists the state after every node execution, enabling:
- Multi-turn conversations (memory between turns)
- Pause and resume
- Time-travel debugging

In [ ]:
from langgraph.checkpoint.memory import MemorySaver

# In-memory checkpointer (great for development)
memory = MemorySaver()

# Recompile with checkpointer
app_with_memory = graph_builder.compile(checkpointer=memory)

# Each thread_id is an isolated conversation
config = {"configurable": {"thread_id": "alice-session-1"}}

# Turn 1
r1 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="What's the weather in Tokyo?")],
     "next_step": "", "iteration_count": 0},
    config
)
print("Turn 1:", r1["messages"][-1].content[:200])

In [ ]:
# Turn 2 — same config → agent remembers previous turn
r2 = app_with_memory.invoke(
    {"messages": [HumanMessage(content="Now calculate 42 * 7.")],
     "next_step": "", "iteration_count": 0},
    config
)
print("Turn 2:", r2["messages"][-1].content)
print("Total messages in state:", len(r2["messages"]))

In [ ]:
# Inspect the current state snapshot
snapshot = app_with_memory.get_state(config)

print("Current state keys:", list(snapshot.values.keys()))
print("Next nodes to execute:", snapshot.next)
print("Step metadata:", snapshot.metadata)
print("Total messages stored:", len(snapshot.values.get("messages", [])))

---
## 7. Human-in-the-Loop

LangGraph can **pause execution** before or after any node, allowing a human to review, approve, or correct the agent's actions.

In [ ]:
# Compile with interrupt_before=["tools"]
# → agent will PAUSE before executing any tool call
hitl_app = graph_builder.compile(
    checkpointer=memory,
    interrupt_before=["tools"]
)

hitl_config = {"configurable": {"thread_id": "hitl-demo"}}

print("Running agent until tool call interrupt...")
r = hitl_app.invoke(
    {"messages": [HumanMessage(content="Get the weather in London.")],
     "next_step": "", "iteration_count": 0},
    hitl_config
)

print("Agent paused. Pending node:", hitl_app.get_state(hitl_config).next)

In [ ]:
# Inspect what the agent wants to do (the pending tool call)
state = hitl_app.get_state(hitl_config)
last_msg = state.values["messages"][-1]

print("Pending action — Tool calls:")
if hasattr(last_msg, 'tool_calls') and last_msg.tool_calls:
    for tc in last_msg.tool_calls:
        print(f"  Tool: {tc['name']}")
        print(f"  Args: {tc['args']}")
else:
    print("  No tool calls pending")

In [ ]:
# Human approves → resume by passing None as input
print("Human approved ✅ — resuming execution...")
final = hitl_app.invoke(None, hitl_config)
print("Final answer:", final["messages"][-1].content)

In [ ]:
# Alternatively — human can MODIFY the state before resuming
hitl_config2 = {"configurable": {"thread_id": "hitl-override"}}

# Start the agent
hitl_app.invoke(
    {"messages": [HumanMessage(content="Get weather for Singapore.")],
     "next_step": "", "iteration_count": 0},
    hitl_config2
)

# Human overrides — update state with a correction
hitl_app.update_state(
    hitl_config2,
    {"messages": [HumanMessage(content="Actually check Tokyo weather instead.")]},
    as_node="agent"
)

print("State updated by human ✅")
final2 = hitl_app.invoke(None, hitl_config2)
print("Corrected answer:", final2["messages"][-1].content)

---
## 8. Streaming Agent Outputs

Stream state updates in real time for responsive UIs and monitoring.

In [ ]:
# stream_mode="values" → full state emitted after every node
print("Streaming with mode='values':\n")
for state in app_with_memory.stream(
    {"messages": [HumanMessage(content="What is 123 * 456?")],
     "next_step": "", "iteration_count": 0},
    {"configurable": {"thread_id": "stream-values"}},
    stream_mode="values"
):
    last = state["messages"][-1]
    role = type(last).__name__
    content = last.content if isinstance(last.content, str) else "[tool call]"
    print(f"  [{role}]: {content[:120]}")

In [ ]:
# stream_mode="updates" → only the delta (changed keys) per node
print("Streaming with mode='updates':\n")
for node_name, update in app_with_memory.stream(
    {"messages": [HumanMessage(content="Get weather for Singapore and London.")],
     "next_step": "", "iteration_count": 0},
    {"configurable": {"thread_id": "stream-updates"}},
    stream_mode="updates"
):
    changed_keys = list(update.keys())
    print(f"  Node '{node_name}' → updated keys: {changed_keys}")

In [ ]:
import asyncio

# Token-level streaming with astream_events
async def stream_tokens():
    print("Token-level streaming:\n")
    async for event in app_with_memory.astream_events(
        {"messages": [HumanMessage(content="Write a haiku about AI.")],
         "next_step": "", "iteration_count": 0},
        {"configurable": {"thread_id": "token-stream"}},
        version="v2"
    ):
        if event["event"] == "on_chat_model_stream":
            chunk = event["data"]["chunk"]
            if chunk.content:
                print(chunk.content, end="", flush=True)
    print()  # newline at end

# Run async function
await stream_tokens()

---
## 9. Multi-Agent Supervisor Pattern

LangGraph excels at orchestrating multiple specialised agents under a **supervisor** that decides who acts next.

In [ ]:
from typing import TypedDict, Annotated, Sequence, Literal
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import StateGraph, START, END
import operator

# ── Multi-Agent State ─────────────────────────────────
class MultiAgentState(TypedDict):
    messages: Annotated[Sequence[BaseMessage], operator.add]
    next_worker: str
    research_output: str
    final_report: str

members = ["researcher", "writer", "FINISH"]

# ── Supervisor Node ───────────────────────────────────
supervisor_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a supervisor managing a team: {members}.\n"
     "Given the conversation, decide who should act next.\n"
     "Reply ONLY with one of: researcher, writer, FINISH"),
    ("human", "{messages}")
])

def supervisor_node(state: MultiAgentState) -> dict:
    chain = supervisor_prompt | model
    msgs_text = "\n".join([
        f"{type(m).__name__}: {m.content[:100]}"
        for m in state["messages"]
    ])
    response = chain.invoke({"members": members, "messages": msgs_text})
    decision = response.content.strip()
    # Normalise
    if "writer" in decision.lower():
        next_w = "writer"
    elif "finish" in decision.lower() or state.get("final_report"):
        next_w = "FINISH"
    else:
        next_w = "researcher"
    return {"next_worker": next_w}

# ── Researcher Node ───────────────────────────────────
def researcher_node(state: MultiAgentState) -> dict:
    last_msg = state["messages"][-1].content
    research = model.invoke(
        f"Research and summarise (2-3 sentences): {last_msg}"
    )
    return {
        "messages": [AIMessage(content=f"[Researcher]: {research.content}")],
        "research_output": research.content
    }

# ── Writer Node ───────────────────────────────────────
def writer_node(state: MultiAgentState) -> dict:
    research = state.get("research_output", "No research available.")
    report = model.invoke(
        f"Write a short professional report (3-4 sentences) based on this research:\n{research}"
    )
    return {
        "messages": [AIMessage(content=f"[Writer]: {report.content}")],
        "final_report": report.content
    }

print("✅ Multi-agent nodes defined")

In [ ]:
from langchain_core.messages import AIMessage

# Build the supervisor graph
multi_graph = StateGraph(MultiAgentState)

multi_graph.add_node("supervisor", supervisor_node)
multi_graph.add_node("researcher", researcher_node)
multi_graph.add_node("writer", writer_node)

# All workers report back to supervisor
multi_graph.add_edge(START, "supervisor")
multi_graph.add_edge("researcher", "supervisor")
multi_graph.add_edge("writer", "supervisor")

# Supervisor routes to workers or END
multi_graph.add_conditional_edges(
    "supervisor",
    lambda state: state["next_worker"],
    {"researcher": "researcher", "writer": "writer", "FINISH": END}
)

multi_app = multi_graph.compile(checkpointer=MemorySaver())

print("\nMulti-agent graph:")
print(multi_app.get_graph().draw_ascii())

In [ ]:
# Run the multi-agent system
result = multi_app.invoke(
    {"messages": [HumanMessage(content="Explain what LangGraph is and why it matters.")],
     "next_worker": "", "research_output": "", "final_report": ""},
    {"configurable": {"thread_id": "multi-agent-run-1"}}
)

print("\n📋 FINAL REPORT:")
print("=" * 60)
print(result.get("final_report", "Report not generated"))

---
## 10. Prebuilt ReAct Agent — `create_react_agent`

For standard use cases, LangGraph ships a ready-to-use ReAct agent — no need to wire the graph manually.

In [ ]:
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver

# One-liner agent with tools and memory
react_agent = create_react_agent(
    model=model,
    tools=[get_weather, calculate],
    checkpointer=MemorySaver(),
    state_modifier="You are a helpful assistant. Always use tools when asked for weather or calculations."
)

config = {"configurable": {"thread_id": "react-demo"}}

result = react_agent.invoke(
    {"messages": [HumanMessage(content="What's the weather in Singapore and calculate 18 * 6?")]},
    config
)
print(result["messages"][-1].content)

In [ ]:
# Demonstrate persistent memory across turns
r2 = react_agent.invoke(
    {"messages": [HumanMessage(content="What was the first city I asked about?")]},
    config  # same thread_id → remembers previous turn
)
print(r2["messages"][-1].content)

---
## 11. Debugging & Observability

LangGraph provides multiple ways to inspect, debug, and replay agent executions.

In [ ]:
# ── Inspect Graph Structure ───────────────────────────
graph_info = app_with_memory.get_graph()

print("Nodes:", list(graph_info.nodes.keys()))
print("Edges:", [(e.source, e.target) for e in graph_info.edges])

In [ ]:
# ── Time Travel — View All Past States ───────────────
config_for_history = {"configurable": {"thread_id": "alice-session-1"}}

print("State history (time travel):")
print("=" * 60)
for i, past_state in enumerate(app_with_memory.get_state_history(config_for_history)):
    step = past_state.metadata.get("step", "?")
    n_messages = len(past_state.values.get("messages", []))
    next_node = past_state.next
    print(f"Step {step}: {n_messages} messages | next: {next_node}")
    if i >= 5:  # limit output
        print("  ... (more history available)")
        break

In [ ]:
# ── Enable LangSmith Tracing ──────────────────────────
# Uncomment and set your LangSmith API key to enable full tracing

# import os
# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_API_KEY"] = "your-langsmith-api-key"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-Demo"
# 
# # Every invoke / stream call is now automatically traced in LangSmith
# result = app_with_memory.invoke({...}, config)

print("💡 Tip: Enable LangSmith tracing for full observability.")
print("   Set LANGCHAIN_TRACING_V2=true and LANGCHAIN_API_KEY in your environment.")

In [ ]:
# ── Export Graph as Mermaid Diagram ──────────────────
mermaid_code = app_with_memory.get_graph().draw_mermaid()
print("Mermaid diagram code:")
print(mermaid_code)

# You can paste this into https://mermaid.live to visualise it!

---
## 🎓 Summary — Key Concepts

| Concept | Description |
|---|---|
| `StateGraph` | Graph container with a typed shared state schema |
| `TypedDict` | Defines the state schema with type safety |
| `Annotated[list, operator.add]` | Reducer — append to lists instead of overwrite |
| **Node** | Python function: `(state) → partial state update` |
| **Direct Edge** | `add_edge(A, B)` — always go from A to B |
| **Conditional Edge** | `add_conditional_edges` — router function decides next node |
| **Cyclic Edge** | Edge back to earlier node — enables iterative agent loops |
| `MemorySaver` | In-memory checkpointer for persistent multi-turn state |
| `interrupt_before` | Pause execution before a node for human review |
| `update_state` | Human can modify state before resuming |
| `get_state_history` | Replay all past state snapshots (time travel) |
| `create_react_agent` | Prebuilt ReAct agent — tools + memory in one line |
| Multi-agent | Supervisor + sub-agent pattern with `conditional_edges` |

---

**💡 Next Steps:**
- Integrate with **LangSmith** for production observability
- Use **SQLite or Redis checkpointer** for persistent storage
- Explore **parallel node execution** with the `Send()` API
- Build a **RAG agent** with vector database tools

---
*Session 4 — Presentation by Dileep*